# Train YOLO11 Nano Piece-Type Segmentation

This notebook trains `yolo11n-seg.pt` on synthetic 256x256 cube images and grayscale masks. The target classes are `center`, `edge`, and `corner`.

Expected uploaded dataset layout:

```text
synthetic_cube_dataset.zip
  images/*.png
  masks/*.png
  metadata.json  # optional
```

The image and mask filenames should share the same stem, for example `images/000123.png` and `masks/000123.png`.

Order randomization is handled explicitly: paired samples are shuffled with a fixed seed before splitting, then copied into randomized train/val filenames.

In [ ]:
!pip -q install ultralytics opencv-python-headless

In [ ]:
from pathlib import Path
import json
import random
import shutil
import zipfile

import cv2
import matplotlib.pyplot as plt
import numpy as np
from google.colab import files
from PIL import Image

WORKDIR = Path('/content/cube_piece_seg')
RAW_DIR = WORKDIR / 'raw'
DATASET_DIR = WORKDIR / 'dataset'

for directory in [RAW_DIR, DATASET_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(WORKDIR)

## Upload Synthetic Dataset Zip

In [ ]:
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('Upload synthetic_cube_dataset.zip')

for name, data in uploaded.items():
    zip_path = RAW_DIR / name
    zip_path.write_bytes(data)
    if zip_path.suffix.lower() == '.zip':
        with zipfile.ZipFile(zip_path) as archive:
            archive.extractall(RAW_DIR)
        print('extracted', zip_path)
    else:
        print('uploaded non-zip', zip_path)

## Configure Instance Mask Values

Use unique nonzero grayscale values per physical piece so adjacent pieces of the same class do not merge. The notebook converts each distinct mask value into one YOLO segmentation instance, then maps that instance to `center`, `edge`, or `corner`.

For easy visual verification, the default values are spread across the full 1-255 range:

- centers: `20, 40, 60, 80, 100, 120`
- edges: `130, 140, ..., 240`
- corners: `248, 249, ..., 255`

Only these configured values are converted to YOLO labels. Other grayscale values are ignored by default; set `STRICT_MASK_VALUES = True` if you want the notebook to fail on unmapped values.


In [ ]:
CLASS_NAMES = ['center', 'edge', 'corner']

# Leave as None to use this full-range piece-instance layout.
# Or set manually, for example: {20: 0, 40: 0, 130: 1, 140: 1, 248: 2}
MASK_VALUE_TO_CLASS = None
DEFAULT_INSTANCE_VALUES = {
    'center': [20, 40, 60, 80, 100, 120],
    'edge': [130, 140, 150, 160, 170, 180, 190, 200, 210, 220, 230, 240],
    'corner': [248, 249, 250, 251, 252, 253, 254, 255],
}

MIN_CONTOUR_AREA = 8
STRICT_MASK_VALUES = False
VAL_FRACTION = 0.2
SEED = 17
IMGSZ = 256


## Build YOLO Segmentation Layout

In [ ]:
from ultralytics.data.converter import convert_segment_masks_to_yolo_seg

def find_dataset_dirs(root: Path):
    image_dirs = [p for p in root.rglob('images') if p.is_dir()]
    mask_dirs = [p for p in root.rglob('masks') if p.is_dir()]
    if not image_dirs or not mask_dirs:
        raise RuntimeError('Could not find images/ and masks/ directories under uploaded zip')
    return image_dirs[0], mask_dirs[0]

def default_mask_value_to_class():
    mapping = {}
    for class_id, class_name in enumerate(CLASS_NAMES):
        for value in DEFAULT_INSTANCE_VALUES[class_name]:
            mapping[int(value)] = class_id
    return mapping

def collect_mask_values(mask_paths, sample_count=50):
    counts = {}
    for mask_path in mask_paths[:sample_count]:
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        values, value_counts = np.unique(mask, return_counts=True)
        for value, count in zip(values.tolist(), value_counts.tolist()):
            counts[int(value)] = counts.get(int(value), 0) + int(count)
    return counts

def audit_raw_masks(mask_paths):
    audit = {
        'images': len(mask_paths),
        'segments': 0,
        'class_segments': {name: 0 for name in CLASS_NAMES},
        'value_segments': {},
        'ignored_values': {},
        'empty_masks': [],
    }
    for mask_path in mask_paths:
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise RuntimeError(f'Could not read mask: {mask_path}')

        mask_segment_count = 0
        for value in sorted(int(v) for v in np.unique(mask) if int(v) != 0):
            binary = (mask == value).astype(np.uint8) * 255
            contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            valid_contours = [contour for contour in contours if cv2.contourArea(contour) >= MIN_CONTOUR_AREA]
            segment_count = len(valid_contours)
            if segment_count == 0:
                continue

            if value in MASK_VALUE_TO_CLASS:
                class_name = CLASS_NAMES[MASK_VALUE_TO_CLASS[value]]
                audit['segments'] += segment_count
                audit['class_segments'][class_name] += segment_count
                audit['value_segments'][value] = audit['value_segments'].get(value, 0) + segment_count
                mask_segment_count += segment_count
            else:
                audit['ignored_values'][value] = audit['ignored_values'].get(value, 0) + segment_count

        if mask_segment_count == 0:
            audit['empty_masks'].append(mask_path.name)
    return audit

def write_converter_mask(source_mask_path: Path, target_mask_path: Path, mask_value_to_converter_value: dict[int, int]):
    mask = cv2.imread(str(source_mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise RuntimeError(f'Could not read mask: {source_mask_path}')

    converter_mask = np.zeros_like(mask, dtype=np.uint8)
    for source_value, converter_value in mask_value_to_converter_value.items():
        converter_mask[mask == source_value] = converter_value
    cv2.imwrite(str(target_mask_path), converter_mask)

def remap_converted_label_file(label_path: Path, converter_class_to_target_class: dict[int, int]):
    if not label_path.exists():
        label_path.write_text('', encoding='utf-8')
        return []

    remapped_lines = []
    for line in label_path.read_text().splitlines():
        parts = line.split()
        if len(parts) < 7 or (len(parts) - 1) % 2 != 0:
            continue
        converter_class = int(float(parts[0]))
        target_class = converter_class_to_target_class.get(converter_class)
        if target_class is None:
            continue
        remapped_lines.append(' '.join([str(target_class), *parts[1:]]))

    label_path.write_text('\n'.join(remapped_lines) + ('\n' if remapped_lines else ''), encoding='utf-8')
    return remapped_lines

images_source, masks_source = find_dataset_dirs(RAW_DIR)
image_paths = sorted([p for p in images_source.glob('*') if p.suffix.lower() in {'.png', '.jpg', '.jpeg'}])
mask_by_stem = {p.stem: p for p in masks_source.glob('*') if p.suffix.lower() in {'.png', '.jpg', '.jpeg'}}
pairs = [(image_path, mask_by_stem[image_path.stem]) for image_path in image_paths if image_path.stem in mask_by_stem]

if not pairs:
    raise RuntimeError('No image/mask pairs found')

if MASK_VALUE_TO_CLASS is None:
    MASK_VALUE_TO_CLASS = default_mask_value_to_class()

observed_counts = collect_mask_values([mask_path for _, mask_path in pairs])
observed_nonzero = sorted(value for value in observed_counts if value != 0)
configured_values = sorted(MASK_VALUE_TO_CLASS)
ignored_values = [value for value in observed_nonzero if value not in MASK_VALUE_TO_CLASS]
missing_configured_values = [value for value in configured_values if value not in observed_nonzero]
if ignored_values and STRICT_MASK_VALUES:
    raise RuntimeError(
        'Observed mask values not covered by MASK_VALUE_TO_CLASS: '
        f'{ignored_values}. Update the mapping cell or set STRICT_MASK_VALUES = False.'
    )

print('Observed nonzero mask values:', observed_nonzero)
print('Configured label values:', configured_values)
if ignored_values:
    print('Ignoring unmapped nonzero values:', ignored_values)
if missing_configured_values:
    print('Configured values not observed in sampled masks:', missing_configured_values)
print('MASK_VALUE_TO_CLASS:', MASK_VALUE_TO_CLASS)

raw_audit = audit_raw_masks([mask_path for _, mask_path in pairs])
print('Raw mask segmentation audit:')
print(json.dumps(raw_audit, indent=2))

mask_value_to_converter_value = {value: index + 1 for index, value in enumerate(configured_values)}
converter_class_to_target_class = {
    converter_value - 1: MASK_VALUE_TO_CLASS[mask_value]
    for mask_value, converter_value in mask_value_to_converter_value.items()
}
print('mask value -> temporary converter value:', mask_value_to_converter_value)
print('converter class -> YOLO target class:', converter_class_to_target_class)

random.Random(SEED).shuffle(pairs)
val_count = max(1, round(len(pairs) * VAL_FRACTION)) if len(pairs) > 1 else 0
splits = {'val': pairs[:val_count], 'train': pairs[val_count:]}

if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
for split in ['train', 'val']:
    (DATASET_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (DATASET_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

converter_masks_dir = DATASET_DIR / '_converter_masks'
conversion_manifest = []
summary = {}
for split, split_pairs in splits.items():
    split_converter_masks_dir = converter_masks_dir / split
    split_converter_masks_dir.mkdir(parents=True, exist_ok=True)

    for index, (image_path, mask_path) in enumerate(split_pairs):
        target_name = f'{split}_{index:06d}{image_path.suffix.lower()}'
        target_image = DATASET_DIR / 'images' / split / target_name
        target_mask = split_converter_masks_dir / f'{Path(target_name).stem}.png'
        shutil.copy2(image_path, target_image)
        write_converter_mask(mask_path, target_mask, mask_value_to_converter_value)
        conversion_manifest.append({
            'split': split,
            'image': str(target_image),
            'mask': str(mask_path),
            'converter_mask': str(target_mask),
            'label': str(DATASET_DIR / 'labels' / split / f'{Path(target_name).stem}.txt'),
        })

    convert_segment_masks_to_yolo_seg(
        masks_dir=str(split_converter_masks_dir),
        output_dir=str(DATASET_DIR / 'labels' / split),
        classes=len(configured_values),
    )

for item in conversion_manifest:
    label_path = Path(item['label'])
    lines = remap_converted_label_file(label_path, converter_class_to_target_class)
    item['segments'] = len(lines)

for split in ['train', 'val']:
    split_items = [item for item in conversion_manifest if item['split'] == split]
    class_counts = {name: 0 for name in CLASS_NAMES}
    for item in split_items:
        label_path = Path(item['label'])
        for line in label_path.read_text().splitlines():
            class_counts[CLASS_NAMES[int(line.split()[0])]] += 1
    summary[split] = {
        'images': len(split_items),
        'segments': sum(item['segments'] for item in split_items),
        'class_segments': class_counts,
    }

manifest_path = DATASET_DIR / 'conversion_manifest.json'
manifest_path.write_text(json.dumps(conversion_manifest, indent=2), encoding='utf-8')

data_yaml = DATASET_DIR / 'data.yaml'
data_yaml.write_text(
    'path: ' + str(DATASET_DIR) + '\n'
    'train: images/train\n'
    'val: images/val\n'
    'names:\n'
    + ''.join(f'  {i}: {name}\n' for i, name in enumerate(CLASS_NAMES)),
    encoding='utf-8',
)

print(json.dumps(summary, indent=2))
print(data_yaml.read_text())


## Sanity Check Converted Segments

This shows the source RGB image, the raw grayscale mask, and the converted YOLO polygons. If the raw mask looks right but the polygon overlay is empty, the issue is in mask value mapping or contour conversion.


In [ ]:
def draw_yolo_segments(image_path: Path, label_path: Path):
    image = np.array(Image.open(image_path).convert('RGB'))
    height, width = image.shape[:2]
    overlay = image.copy()
    colors = [(255, 80, 80), (80, 255, 120), (80, 160, 255)]
    line_count = 0
    if label_path.exists():
        for line in label_path.read_text().splitlines():
            parts = line.split()
            if len(parts) < 7 or (len(parts) - 1) % 2 != 0:
                print('Skipping malformed label line:', line)
                continue
            class_id = int(parts[0])
            coords = np.array([float(v) for v in parts[1:]], dtype=np.float32).reshape(-1, 2)
            pts = np.column_stack([coords[:, 0] * width, coords[:, 1] * height]).astype(np.int32)
            cv2.fillPoly(overlay, [pts], colors[class_id])
            cv2.polylines(overlay, [pts], True, colors[class_id], 2)
            line_count += 1
    blended = cv2.addWeighted(image, 0.60, overlay, 0.40, 0)
    return blended, line_count

def colorize_mask(mask_path: Path):
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise RuntimeError(f'Could not read mask: {mask_path}')
    return cv2.applyColorMap(mask, cv2.COLORMAP_VIRIDIS)

manifest = json.loads((DATASET_DIR / 'conversion_manifest.json').read_text())
train_items = [item for item in manifest if item['split'] == 'train'][:4]

plt.figure(figsize=(12, 4 * len(train_items)))
for row, item in enumerate(train_items):
    image_path = Path(item['image'])
    mask_path = Path(item['mask'])
    label_path = Path(item['label'])
    overlay, line_count = draw_yolo_segments(image_path, label_path)

    plt.subplot(len(train_items), 3, row * 3 + 1)
    plt.imshow(Image.open(image_path).convert('RGB'))
    plt.axis('off')
    plt.title(f'RGB: {image_path.name}', fontsize=8)

    plt.subplot(len(train_items), 3, row * 3 + 2)
    plt.imshow(cv2.cvtColor(colorize_mask(mask_path), cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title(f'raw mask values: {sorted(np.unique(cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)).tolist())}', fontsize=8)

    plt.subplot(len(train_items), 3, row * 3 + 3)
    plt.imshow(overlay)
    plt.axis('off')
    plt.title(f'YOLO polygons: {line_count}', fontsize=8)
plt.tight_layout()


## Train YOLO11 Nano Segmentation

In [ ]:
from ultralytics import YOLO

MODEL_NAME = 'yolo11n-seg.pt'
EPOCHS = 100
BATCH = -1

model = YOLO(MODEL_NAME)
results = model.train(
    data=str(DATASET_DIR / 'data.yaml'),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    project=str(WORKDIR / 'runs'),
    name='cube_piece_yolo11n_seg',
    patience=25,
    cache=True,
    plots=True,
)

## Validate Best Weights

In [ ]:
best_model_path = WORKDIR / 'runs' / 'cube_piece_yolo11n_seg' / 'weights' / 'best.pt'
print('best model:', best_model_path)

best_model = YOLO(str(best_model_path))
metrics = best_model.val(data=str(DATASET_DIR / 'data.yaml'), imgsz=IMGSZ, device=0)
print(metrics)

## Download Weights

In [ ]:
files.download(str(best_model_path))